# Sampling important IMAGE tokens on SmolVLM2 — multiple images & questions

For each (image, question) it runs the full pipeline on real
**`HuggingFaceTB/SmolVLM2-2.2B-Instruct`**:

```
raw text->vision attention -> rater_selection (important TEXT) -> visual_selection (important IMAGE, all layers)
```

and visualises, per pair: the image, the image-token importance heatmap, and the
kept patches at **pct = 0.5 / 0.75 / 0.9** (top 41 / 21 / 9 of 81).

Includes the cats image asked TWO ways — about the **cats** and about the
**remote** — to see whether the selected patches move to the asked object.

> **Runtime:** GPU runtime. SmolVLM2 is open (no token needed).

## 1. Install + clone

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate huggingface_hub safetensors pillow num2words pytest matplotlib

In [ ]:
!rm -rf text_vision_attention_map          # fresh checkout
!git clone -q https://github.com/shubhamOjha1000/text_vision_attention_map.git
%cd text_vision_attention_map

## 2. Setup: load modules + define the (image, question) pairs and helpers
The model is loaded once and reused across all pairs.

In [ ]:
import importlib.util, os, math
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def _load(mod, rel):
    spec = importlib.util.spec_from_file_location(mod, os.path.join(os.getcwd(), rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

S = _load("probe_smolvlm", "tests/probe_smolvlm.py")
V = _load("test_visual_selection", "tests/test_visual_selection.py")
import rater_selection as RS
import visual_selection as VS

from transformers import AutoProcessor
tokenizer = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM2-2.2B-Instruct").tokenizer

# (image URL, question).  The cats image appears twice: cats vs remote.
PAIRS = [
    ("http://images.cocodataset.org/val2017/000000039769.jpg", "How many cats are in the image?"),
    ("http://images.cocodataset.org/val2017/000000039769.jpg", "Where is the remote control?"),
    ("https://ultralytics.com/images/bus.jpg",                 "Where is the bus?"),
    ("https://ultralytics.com/images/zidane.jpg",              "How many people are in the image?"),
]
PCTS = (0.5, 0.75, 0.9)          # image-token drop fractions to visualise

def to_grid(vec, L_v):
    g = int(round(math.sqrt(L_v)))
    if g * g != L_v:
        g = math.ceil(math.sqrt(L_v)); vec = np.concatenate([vec, np.zeros(g * g - L_v, vec.dtype)])
    return vec.reshape(g, g)

def up(a, size, mode=Image.BILINEAR):
    a = (a / (a.max() + 1e-9) * 255).astype('uint8')
    return np.array(Image.fromarray(a).resize(size, mode))

def probe_and_select(url, question):
    img = S.load_image(url)
    o = S.make_smolvlm_output(image=img, question=question)   # all decoder layers
    if o is None:
        return None
    maps, tpos, vpos = RS.sliced_maps_from_full(o.raw_scores, o.image_token_mask, o.text_token_mask)
    text_tokens = tokenizer.convert_ids_to_tokens(o.input_ids[tpos].tolist())
    rres = RS.select_important_text_tokens(maps, text_tokens=text_tokens, tokenizer=tokenizer,
                                           question=question, pct=0.5)
    vres = {p: VS.select_from_rater(maps, rres, pct=p) for p in PCTS}
    return dict(img=img, question=question, raters=rres.kept_tokens(text_tokens), vres=vres)

print("pairs:", len(PAIRS), "| pct levels:", PCTS)

## 3. Run all pairs and visualise
Rows = (image, question). Columns = image | importance heatmap | kept @0.5 | @0.75 | @0.9.
First run downloads the ~4.5 GB weights.

In [ ]:
rows = [probe_and_select(u, q) for (u, q) in PAIRS]
rows = [r for r in rows if r is not None]
assert rows, "no pairs produced a result (SmolVLM probe failed to load?)"

ncols = 2 + len(PCTS)
fig, axes = plt.subplots(len(rows), ncols, figsize=(3.6 * ncols, 3.6 * len(rows)))
if len(rows) == 1:
    axes = axes[None, :]

for r, row in enumerate(rows):
    img = row["img"]
    vr0 = row["vres"][PCTS[0]]
    L_v = vr0.L_v
    heat = to_grid(vr0.importance.numpy(), L_v)

    axes[r, 0].imshow(img); axes[r, 0].axis("off")
    axes[r, 0].set_title(f"Q: {row['question']}\nraters: {row['raters']}", fontsize=8)

    axes[r, 1].imshow(img); axes[r, 1].imshow(up(heat, img.size), cmap="jet", alpha=0.5)
    axes[r, 1].set_title("importance (all layers)", fontsize=9); axes[r, 1].axis("off")

    for c, p in enumerate(PCTS):
        vr = row["vres"][p]
        km = to_grid(vr.vision_mask.numpy().astype(np.float32), L_v)
        axes[r, 2 + c].imshow(img)
        axes[r, 2 + c].imshow(up(km, img.size, Image.NEAREST), cmap="Greens", alpha=0.5)
        axes[r, 2 + c].set_title(f"kept top {vr.n_kept}/{L_v}  (pct={p})", fontsize=9)
        axes[r, 2 + c].axis("off")

plt.tight_layout(); plt.show()

## 4. Numbers per pair (raters + kept counts)

In [ ]:
for row in rows:
    counts = {p: row['vres'][p].n_kept for p in PCTS}
    L_v = row['vres'][PCTS[0]].L_v
    print(f"Q: {row['question']}")
    print(f"   raters       : {row['raters']}")
    print(f"   layers used  : {len(row['vres'][PCTS[0]].band)}  | L_v = {L_v}")
    print(f"   kept image tokens : " + "  ".join(f"pct={p}->{counts[p]}" for p in PCTS))
    print()

## 5. (Optional) invariant checks on the first pair

In [ ]:
img0 = S.load_image(PAIRS[0][0])
o0 = S.make_smolvlm_output(image=img0, question=PAIRS[0][1])
maps0, tpos0, _ = RS.sliced_maps_from_full(o0.raw_scores, o0.image_token_mask, o0.text_token_mask)
tt0 = tokenizer.convert_ids_to_tokens(o0.input_ids[tpos0].tolist())
rres0 = RS.select_important_text_tokens(maps0, text_tokens=tt0, tokenizer=tokenizer,
                                        question=PAIRS[0][1], pct=0.5)
case0 = V.make_case(maps0, rres0.rater_mask, pct=0.5)
passed = 0
for fn in V.ALL_CHECKS:
    try:
        fn(case0); print(f"PASS  {fn.__name__}"); passed += 1
    except AssertionError as e:
        print(f"FAIL  {fn.__name__}: {e}")
print(f"\n{passed}/{len(V.ALL_CHECKS)} visual-selection checks passed.")

## 6. Kill sink tokens — baseline subtraction (B) + drop invariant sinks (A)

The importance peaked on a fixed background patch **regardless of the question**
(a positional *sink*). Fix:
- **B**: build a `baseline` = mean importance over several questions on the SAME
  image (the question-invariant part = the sink), then subtract it.
- **A**: additionally zero the top-`k` baseline (sink) patches.

Then compare **before vs after** for the cats-vs-remote questions — the debiased
map should finally *move* to the asked object.

In [ ]:
# --- build the position-bias baseline from several questions on the CATS image ---
CAT_URL = PAIRS[0][0]
cat_img = S.load_image(CAT_URL)
BASELINE_QS = [
    "How many cats are in the image?",
    "Where is the remote control?",
    "What is in the image?",
    "Describe the picture.",
]

def context_for(question):
    o = S.make_smolvlm_output(image=cat_img, question=question)
    maps, tpos, _ = RS.sliced_maps_from_full(o.raw_scores, o.image_token_mask, o.text_token_mask)
    tt = tokenizer.convert_ids_to_tokens(o.input_ids[tpos].tolist())
    rr = RS.select_important_text_tokens(maps, text_tokens=tt, tokenizer=tokenizer,
                                         question=question, pct=0.5)
    imp, _, _, _ = VS.image_importance(maps, rr.rater_mask)
    return dict(question=question, maps=maps, rmask=rr.rater_mask,
                importance=imp, raters=rr.kept_tokens(tt))

ctx = [context_for(q) for q in BASELINE_QS]
baseline = VS.make_baseline([c["importance"] for c in ctx])     # B: sink = invariant part
print("baseline (sink) peak patch index:", int(baseline.argmax()),
      "  (question-invariant -> a sink)")

DROP_K = 3            # A: also hard-drop the top-3 sink patches
for c in ctx[:2]:     # cats, remote
    c["debiased"] = VS.select_debiased(c["maps"], c["rmask"],
                                       baseline=baseline, drop_sink_k=DROP_K, pct=0.9)
    print(f"Q: {c['question']:<34} raw-peak={int(c['importance'].argmax()):>2}  "
          f"debiased-peak={int(c['debiased'].importance.argmax()):>2}")

In [ ]:
# --- before vs after: raw importance | debiased importance | debiased kept (top-9) ---
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for r, c in enumerate(ctx[:2]):            # row 0 = cats, row 1 = remote
    L_v = c["importance"].numel()
    raw_h = to_grid(c["importance"].numpy(), L_v)
    deb_h = to_grid(c["debiased"].importance.numpy(), L_v)
    keep = to_grid(c["debiased"].vision_mask.numpy().astype(np.float32), L_v)

    axes[r, 0].imshow(cat_img); axes[r, 0].axis("off")
    axes[r, 0].set_title(f"Q: {c['question']}\nraters: {c['raters']}", fontsize=8)
    axes[r, 1].imshow(cat_img); axes[r, 1].imshow(up(raw_h, cat_img.size), cmap="jet", alpha=0.5)
    axes[r, 1].set_title("raw importance (sink)", fontsize=9); axes[r, 1].axis("off")
    axes[r, 2].imshow(cat_img); axes[r, 2].imshow(up(deb_h, cat_img.size), cmap="jet", alpha=0.5)
    axes[r, 2].set_title("debiased (B + A)", fontsize=9); axes[r, 2].axis("off")
    axes[r, 3].imshow(cat_img)
    axes[r, 3].imshow(up(keep, cat_img.size, Image.NEAREST), cmap="Greens", alpha=0.5)
    axes[r, 3].set_title(f"debiased kept top {c['debiased'].n_kept}", fontsize=9); axes[r, 3].axis("off")
plt.tight_layout(); plt.show()